# 01 — Data exploration and protocol (Phase 0)

Validates the 15 base + 5 novel category protocol, builds fixed train / val-tune / val-test image-ID manifests, and writes slim COCO annotation JSONs. Nothing is trained here.

Before running, copy your two annotation JSONs into `data/annotations/`:
- `instances_train2017.json`
- `instances_val2017.json`


In [ ]:
from pathlib import Path
import yaml, json
from pprint import pprint

cfg = yaml.safe_load(open("configs/config.yaml"))
print("Project:", cfg["project"]["name"], "| device:", cfg["project"]["device"])
print()
print("Selected classes:")
pprint(cfg["selected_classes"])


In [ ]:
%run scripts/prepare_data.py --config configs/config.yaml


In [ ]:
import json
from pprint import pprint

meta = json.load(open("data/splits/split_metadata.json"))
print("Split metadata:")
pprint(meta)

counts = json.load(open("data/metadata/category_counts.json"))
print()
print("Category counts (train / val-tune / val-test):")
for name, c in counts.items():
    print(f"  {name:12s} [{c['split']:5s}] train={c['train_instances']:5d} imgs, "
          f"tune={c['val_tune_instances']:4d} imgs, test={c['val_test_instances']:4d} imgs")


In [ ]:
import json
import matplotlib.pyplot as plt

counts = json.load(open("data/metadata/category_counts.json"))
names = list(counts.keys())
train_vals = [counts[n]["train_instances"] for n in names]

plt.figure(figsize=(10, 4))
plt.bar(names, train_vals)
plt.xticks(rotation=45, ha="right")
plt.ylabel("instances")
plt.title("Instances per selected class in the train subset")
plt.tight_layout()
plt.show()


In [ ]:
from pycocotools.coco import COCO
from PIL import Image
import matplotlib.pyplot as plt
from src.openworld.visualization import draw_boxes

coco = COCO("data/annotations/instances_train2017.stripped.json")
train_ids = json.load(open("data/splits/train_image_ids.json"))

for iid in train_ids[:4]:
    img = coco.loadImgs(iid)[0]
    ann_ids = coco.getAnnIds(imgIds=iid)
    anns = coco.loadAnns(ann_ids)
    image = Image.open("data/images/train2017/" + img["file_name"]).convert("RGB")
    boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2],
              a["bbox"][1] + a["bbox"][3]] for a in anns]
    labels = [coco.cats[a["category_id"]]["name"] for a in anns]
    plt.figure(figsize=(5, 5))
    plt.imshow(draw_boxes(image, boxes, labels))
    plt.axis("off")
    plt.title(f"image {iid} — {img['file_name']}")
    plt.show()
